In [ ]:
%pip install pandas

KPI 3: Score par departements

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data_base_copro.csv')
df.columns = df.columns.str.strip().str.lower()

cols_numeriques = ['lots_habitation', 'lots_parking', 'total_lots']
for col in cols_numeriques:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)


df_propre = df[df['lots_habitation'] >= 5].copy()
df_propre['ratio_habitation'] = df_propre['lots_habitation'] / df_propre['total_lots'].replace(0, np.nan)
df_propre = df_propre[df_propre['ratio_habitation'] >= 0.3].copy()


df_unique = df_propre.groupby(['adresse', 'ville', 'code_postal', 'dept_code', 'dept_nom']).agg({
    'lots_habitation': 'sum',
    'lots_parking': 'sum',
    'lat': 'first',
    'long': 'first'
}).reset_index()


df_unique['score_immeuble'] = df_unique['lots_parking'] / df_unique['lots_habitation'].replace(0, 1)
df_final = df_unique[df_unique['score_immeuble'] <= 5].copy()

df_final['dept_code'] = df_final['dept_code'].astype(str).str.replace('.0', '', regex=False)



df_final = df_final[df_final['dept_nom'] != 'non connu']

kpi3_dept = df_final.groupby(['dept_code', 'dept_nom']).agg(
    nb_coproprietes_cibles=('adresse', 'count'), 
    score_potentiel_dept=('score_immeuble', 'mean') 
).reset_index()

kpi3_dept = kpi3_dept[kpi3_dept['nb_coproprietes_cibles'] >= 20].copy()

kpi3_dept['score_potentiel_dept'] = kpi3_dept['score_potentiel_dept'].round(2)

kpi3_dept = kpi3_dept.sort_values('score_potentiel_dept', ascending=False).reset_index(drop=True)

display(kpi3_dept.head(10))

,dept_code,dept_nom,nb_coproprietes_cibles,score_potentiel_dept
0,973,Guyane,72,1.23
1,972,Martinique,400,1.21
2,974,La Réunion,1386,1.19
3,976,Mayotte,31,1.15
4,40,Landes,1297,1.12
5,01,Ain,252,1.10
6,77,Seine-et-Marne,3985,1.07
7,91,Essonne,3711,1.06
8,31,Haute-Garonne,6622,1.04
9,33,Gironde,5247,1.03


Export des données pour l'équipe Front-End

In [ ]:
kpi3_dept.to_csv('export_kpi3_departements.csv', index=False, encoding='utf-8')